<a href="https://colab.research.google.com/github/tteon/seocho/blob/main/tutorials/ontology_guardrail_before_after.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ontology Guardrails: pick the right one for your corpus, then *measure* whether it helps

A **guardrail** in SEOCHO is a curated ontology (a typed schema of classes) that constrains
what an LLM is allowed to extract from your documents. The promise is higher precision and
conformance. But a guardrail is only as good as its *fit to your corpus* — and the only honest
way to know if it helps is to **measure the LLM before vs after applying it**.

This tutorial walks the full loop on a small financial corpus (10 FinDER cases):

1. **Profile** the corpus with an open, schema-free extraction — *what types does it actually need?*
2. **Score** candidate ontologies against that profile with the corpus-aware scorecard.
3. **Select** the best guardrail automatically (`select_guardrail`) — domain-adaptively.
4. **Measure** answer quality BEFORE (open extraction) vs AFTER (guardrailed extraction).
5. **Read the result honestly** — including a case where the guardrail *hurt*, and why the
   scorecard predicted it.

> **Reading levels** — *Basic*: run every cell top to bottom and read the markdown. *Deeper*: the
> callouts link the design decisions (ADR-0116 corpus coverage, ADR-0122 domain-adaptive selection,
> ADR-0141 powered evaluation).

> **Runs with no infrastructure.** The offline core (profile → score → select) is pure Python — no
> graph database, no API key. The before/after measurement ships a **cached result** so the whole
> notebook runs end-to-end as-is; set `MARA_API_KEY` to re-run the LLM steps live.

## 0. Setup — run me first

In [ ]:
# Locate the repo (for datasets + the in-repo SDK), then install the SDK only if absent.
import json, os, sys
from pathlib import Path

def _locate_repo():
    # Local checkout: walk up for the examples/ + datasets markers.
    here = Path.cwd()
    for base in [here, *here.parents]:
        if (base / "examples" / "datasets" / "fibo_plus.jsonld").exists():
            return base
    # Colab / standalone: shallow-clone the repo for the datasets.
    if not Path("seocho").exists():
        os.system("git clone --depth 1 https://github.com/tteon/seocho")
    return Path("seocho")

REPO = _locate_repo()
DATA = REPO / "tutorials" / "data"
sys.path.insert(0, str(REPO / "src"))  # prefer the in-repo SDK when running from a checkout

try:
    import seocho  # noqa: F401
except ImportError:
    %pip install -q seocho
    import seocho  # noqa: F401

print("repo root:", REPO)
print("seocho   :", os.path.dirname(seocho.__file__))

In [2]:
# API key is OPTIONAL — only needed to re-run the LLM before/after live.
# Per SEOCHO's MARA-first default, we read MARA. Leave unset to use cached results.
import getpass
def _maybe_set_env(key):
    if key not in os.environ:
        try:
            val = getpass.getpass(f"{key} (optional — press Enter to skip and use cached results): ")
            if val:
                os.environ[key] = val
        except Exception:
            pass

_maybe_set_env("MARA_API_KEY")
LIVE = bool(os.environ.get("MARA_API_KEY"))
MODEL = "DeepSeek-V3.1"  # MARA, OpenAI-compatible
print("live LLM re-run:", LIVE)

live LLM re-run: False


## 1. The corpus and the questions

We use a 10-case slice of [FinDER](https://huggingface.co/datasets/Linq-AI-Research/FinDER) — real
S&P 500 10-K snippets, each with a question and a reference answer, spanning categories like
*Company Overview, Financials, Governance, Legal, Risk*. A guardrail has to serve **all** of them.

In [3]:
cases = json.loads((REPO / "examples/finder/datasets/finder_tutorial_subset.json").read_text())
print(f"{len(cases)} cases\n")
c0 = cases[0]
print("category :", c0["category"])
print("text     :", c0["text"][:180], "...")
print("question :", c0["question"])
print("reference:", c0["expected_answer"])

10 cases

category : Company Overview
text     : Apple Inc. (AAPL) is a multinational technology company headquartered in Cupertino, California. The company designs, manufactures, and markets smartphones, personal computers, tabl ...
question : Where is Apple Inc. headquartered?
reference: Cupertino, California


## 2. Profile the corpus — what types does it *actually* need?

The trick (ADR-0116): run an **open, schema-free** extraction first. The label frequencies that come
back are independent of any candidate ontology — they describe what the *corpus* demands. SEOCHO's
`build_corpus_profile` aggregates them; `numeric_intensity` summarizes how metric-heavy the corpus is.

We ship the open-extraction graphs so this runs offline; with a key it re-extracts live.

In [4]:
from seocho.ontology_scorecard import build_corpus_profile
from seocho.guardrail_selector import numeric_intensity

if LIVE:
    import re
    from seocho.llm_structured import structured_complete
    from seocho.store.llm import create_llm_backend
    be = create_llm_backend(provider="mara", model=MODEL, api_key=os.environ["MARA_API_KEY"])
    SYS = ("You extract entities from financial text. For each entity give a short GENERAL type "
           "label (Company, Person, FinancialMetric, Location, Product, Regulation, Risk, ...). "
           "Do NOT use a fixed schema. Return ONLY JSON.")
    USR = 'TEXT:\n{doc}\n\nReturn JSON: {{"nodes":[{{"label":"<Type>","name":"..."}}]}}'
    def _extract(text):
        ex = structured_complete(be, system=SYS, user=USR.format(doc=text), model=MODEL,
                                 task_hint="json_extraction")
        return {"nodes": [n for n in ex.get("nodes", []) if isinstance(n, dict) and n.get("label")]}
    open_graphs = [_extract(c["text"]) for c in cases]
else:
    open_graphs = json.loads((DATA / "finder_tutorial_open_graphs.json").read_text())

profile = build_corpus_profile(open_graphs, source="FinDER tutorial subset (open extraction)")
ni = numeric_intensity(profile)
print(f"docs={profile.doc_count}  distinct types={len(profile.label_frequencies)}  numeric_intensity={ni}")
print("\ntop types the corpus needs:")
for label, n in sorted(profile.label_frequencies.items(), key=lambda kv: -kv[1])[:10]:
    print(f"  {label:18s} {n}")

docs=10  distinct types=20  numeric_intensity=0.4

top types the corpus needs:
  FinancialMetric    31
  Company            16
  Product            9
  Person             5
  BusinessSegment    4
  OrganizationType   4
  CurrencyAmount     3
  Percentage         2
  OrganizationalUnit 2
  Regulation         2


`numeric_intensity ≈ 0.4` — below the 0.5 threshold, so this corpus is **entity/qualitative-leaning**,
not purely numeric. That distinction drives the selector in step 4 (ADR-0122: rich vocabulary helps
entity answers, but does *not* help numeric answers — for those the lever is numeric validation, not
more classes). Notice the corpus also needs `Location`, `Product`, `BusinessSegment`, `CurrencyAmount` —
remember those; they matter shortly.

## 3. Score candidate ontologies — the corpus-aware scorecard

We have three curated FIBO-derived guardrail candidates of increasing richness. `score_ontology(...,
profile="guardrail")` grades each across five quality dimensions **plus** `corpus_coverage` — how well
the ontology's vocabulary covers what the corpus needs. The `guardrail` weight profile emphasizes
constraint richness + corpus coverage (the dimensions that predict extraction-guardrail value).

In [5]:
from seocho.ontology import Ontology
from seocho.ontology_scorecard import score_ontology

candidates = {
    "lean": Ontology.from_jsonld(str(REPO / "examples/datasets/fibo_minus.jsonld")),  # 2 classes
    "base": Ontology.from_jsonld(str(REPO / "examples/datasets/fibo_base.jsonld")),   # 4 classes
    "rich": Ontology.from_jsonld(str(REPO / "examples/datasets/fibo_plus.jsonld")),   # 9 classes
}

for name, onto in candidates.items():
    card = score_ontology(onto, corpus_profile=profile, profile="guardrail")
    cov = card.dimension("corpus_coverage").score
    print(f"{name:5s}  grade={card.grade}  overall={card.overall_score:.3f}  "
          f"corpus_coverage={cov:.3f}  ({len(onto.nodes)} classes)")
    gaps = [w.message for w in card.weak_points if w.dimension == "corpus_coverage"][:3]
    for g in gaps:
        print(f"        gap: {g}")

lean   grade=B  overall=0.803  corpus_coverage=0.522  (2 classes)
        gap: Corpus mentions 'Person' 5× (6% of entities) but the ontology has no matching class — add it (or an alias) so the guardrail can represent it.
        gap: Corpus mentions 'Product' 9× (10% of entities) but the ontology has no matching class — add it (or an alias) so the guardrail can represent it.
        gap: Corpus mentions 'BusinessSegment' 4× (4% of entities) but the ontology has no matching class — add it (or an alias) so the guardrail can represent it.
base   grade=B  overall=0.839  corpus_coverage=0.600  (4 classes)
        gap: Corpus mentions 'Product' 9× (10% of entities) but the ontology has no matching class — add it (or an alias) so the guardrail can represent it.
        gap: Corpus mentions 'BusinessSegment' 4× (4% of entities) but the ontology has no matching class — add it (or an alias) so the guardrail can represent it.
        gap: Corpus mentions 'CurrencyAmount' 3× (3% of entities) but t

The scorecard is doing the work an ontology engineer would do by hand: it tells you **exactly which
corpus-needed types each candidate is missing**. Even `rich` (the best, coverage ≈ 0.71) still has gaps —
e.g. it has no `Location` class. Hold that thought.

## 4. Let SEOCHO pick the guardrail — `select_guardrail`

`select_guardrail` combines the coverage scores with the corpus's numeric intensity and applies the
ADR-0122 rule automatically: for an **entity** corpus → the **highest-coverage** candidate; for a
**numeric** corpus → the **leanest adequate** one (and advise numeric validation instead of enrichment).

In [6]:
from seocho.guardrail_selector import select_guardrail

rec = select_guardrail(candidates, profile)
print("chosen          :", rec.chosen)
print("domain_kind     :", rec.domain_kind)
print("numeric_intensity:", rec.numeric_intensity)
print("rationale       :", rec.rationale)
for a in rec.advisories:
    print("advisory        :", a)

chosen          : rich
domain_kind     : entity
numeric_intensity: 0.4
rationale       : entity/qualitative corpus (numeric_intensity=0.4); per ADR-0122 a richer guardrail materially improves answers here, so chose the highest-coverage guardrail 'rich' (coverage 0.7111, 9 classes).


For this entity-leaning corpus the selector picks **`rich`** — the highest-coverage candidate. (On a
numeric-heavy corpus it would instead pick the leanest adequate guardrail and advise numeric
validation; see `examples/finder/` and ADR-0122.)

## 5. Measure: BEFORE vs AFTER

Does the chosen guardrail actually improve answers? We compare, on the same 10 questions:

- **BEFORE** — answer from the *open* extraction (no guardrail).
- **AFTER**  — answer from a *guardrailed* extraction constrained to the chosen ontology's types.

A separate judge call (temperature 0) grades each answer against the reference. This ships a cached
result; set `MARA_API_KEY` to re-run live.

In [7]:
if LIVE:
    chosen_onto = candidates[rec.chosen]
    allowed = ", ".join(chosen_onto.nodes)
    GSYS = ("You extract entities from financial text using ONLY these allowed types: "
            f"[{allowed}]. Discard anything that does not fit. Return ONLY JSON.")
    ASYS = "Answer the question using ONLY the facts in the provided graph context. Be concise."
    JSYS = ('You grade an answer against a reference. Return ONLY JSON {"correct": true|false} — '
            "true iff the answer conveys the reference fact.")
    def _guard(text):
        ex = structured_complete(be, system=GSYS, user=USR.format(doc=text), model=MODEL,
                                 task_hint="json_extraction")
        return {"nodes": [n for n in ex.get("nodes", []) if isinstance(n, dict) and n.get("label")]}
    def _answer(graph, q):
        ctx = "; ".join(f'{n.get("label")}: {n.get("name")}' for n in graph.get("nodes", []))
        return be.complete(system=ASYS, user=f"GRAPH:\n{ctx}\n\nQUESTION: {q}",
                           temperature=0.0, model=MODEL).text.strip()
    def _judge(ans, ref, q):
        v = structured_complete(be, system=JSYS, user=f"QUESTION: {q}\nREFERENCE: {ref}\nANSWER: {ans}",
                                model=MODEL, task_hint="json_extraction")
        return bool(v.get("correct"))
    rows = []
    for i, c in enumerate(cases):
        g_guard = _guard(c["text"])
        a_before = _answer(open_graphs[i], c["question"])
        a_after = _answer(g_guard, c["question"])
        rows.append({"id": c["id"], "category": c["category"], "q": c["question"],
                     "ref": c["expected_answer"], "before": a_before, "after": a_after,
                     "before_ok": _judge(a_before, c["expected_answer"], c["question"]),
                     "after_ok": _judge(a_after, c["expected_answer"], c["question"])})
    before_acc = sum(r["before_ok"] for r in rows) / len(rows)
    after_acc = sum(r["after_ok"] for r in rows) / len(rows)
else:
    cached = json.loads((DATA / "finder_tutorial_eval_cached.json").read_text())
    rows, before_acc, after_acc = cached["rows"], cached["before_acc"], cached["after_acc"]

print(f"BEFORE (open extraction)        accuracy: {before_acc:.2f}")
print(f"AFTER  (guardrailed: '{rec.chosen}')      accuracy: {after_acc:.2f}\n")
print(f"{'category':20s} {'before':7s} {'after':6s} question")
for r in rows:
    print(f"{r['category'][:20]:20s} {'OK' if r['before_ok'] else ' . ':7s} "
          f"{'OK' if r['after_ok'] else ' . ':6s} {r['q'][:46]}")

BEFORE (open extraction)        accuracy: 0.90
AFTER  (guardrailed: 'rich')      accuracy: 0.80

category             before  after  question
Company Overview     OK       .     Where is Apple Inc. headquartered?
Financials           OK      OK     What was Microsoft's total revenue in fiscal y
Shareholder Return   OK      OK     How much did NVIDIA pay in dividends during fi
Governance           OK      OK     Who is the Chair of Alphabet's Board of Direct
Legal                OK      OK     What was the value of Meta's Cambridge Analyti
Accounting           OK       .     What was Tesla's automotive revenue in fiscal 
Footnotes             .      OK     What was JPMorgan Chase's CET1 capital ratio a
Risk                 OK      OK     Which subsidiary of Berkshire Hathaway provide
Company Overview     OK      OK     Who is the CEO of Amazon.com, Inc.?
Financials           OK      OK     What was the fair value of Berkshire Hathaway'


## 6. Read the result honestly

On this run the open baseline scored **0.90** and the guardrailed extraction **0.80** — the guardrail
*lowered* accuracy on a tiny N=10. That is not a failure of the tutorial; it is the honest, important
lesson. Look at the one case that flipped from correct to wrong:

In [8]:
apple = next(r for r in rows if r["id"] == "finder_tut_001")
print("question :", apple["q"])
print("reference:", apple["ref"])
print("\nBEFORE (open) :", apple["before"])
print("\nAFTER (rich)  :", apple["after"])

question : Where is Apple Inc. headquartered?
reference: Cupertino, California

BEFORE (open) : Cupertino, California.

AFTER (rich)  : Based solely on the provided graph context, I cannot determine the headquarters location of Apple Inc. The graph only lists the company and its products, not its headquarters.


*"Where is Apple headquartered?" → "Cupertino, California."* The **open** extraction captured a
`Location` entity, so the BEFORE answer is right. The **`rich`** guardrail has **no `Location` class** —
so the guardrailed extraction *discarded* the headquarters and the AFTER answer is "cannot determine."

This is exactly the gap the scorecard flagged in step 3. The takeaways:

- **A guardrail only helps where its coverage matches the corpus.** Constraining extraction to a schema
  that omits a needed type silently drops facts. The corpus-coverage dimension exists precisely to make
  that risk visible *before* you measure.
- **Coverage is a selection proxy, not a guarantee of answer accuracy.** SEOCHO's powered evaluation
  (ADR-0141, McNemar + bootstrap on N=400, 2-judge consensus) found curated and automated guardrails
  *statistically equivalent on answers* despite coverage differences — and ADR-0122 found enrichment
  helps entity categories but not numeric ones. A ±0.10 delta on N=10 is within noise; **don't conclude
  from small N.** Use a powered run before claiming a win.
- **This is the virtuous cycle.** The scorecard told us `rich` is missing `Location`/`Product`/
  `CurrencyAmount`. Add those classes, re-score (coverage rises), re-measure — and the AFTER number
  should recover. Measuring *is* the feature; the guardrail is only the lever.

## 7. Next steps

- **Close the coverage gaps** the scorecard surfaced (add `Location`, `Product`, …), re-run steps 3–6,
  and watch coverage and AFTER accuracy move together.
- **Run a powered evaluation** instead of N=10: `seocho sweep` over the full FinDER corpus with paired
  McNemar + bootstrap and a 2-judge consensus (see `examples/finder/` and `docs/RUN_SPECS.md`).
- **Declare a guardrail in a run spec** — no Python. An `ontology.select` block hands SEOCHO the
  candidates + a corpus profile and it picks for you; an `ontology.select.fibo` block builds a
  version-pinned, official-FIBO-derived guardrail and selects the best-covering module
  (`docs/RUN_SPECS.md`).
- **Ship the canonical financial corpus profile** (`docs/decisions/ADR-0143-full-corpus.json`, the full
  5,654-doc FinDER profile) as your `corpus_profile` so the selector ranks against real-world demand.

*No infrastructure was created by this notebook — nothing to clean up.*